# 🎵 PySpark Queries for Spotify Gold Layer Data

Notebook này sử dụng PySpark để truy vấn và hiển thị dữ liệu từ Gold layer tương tự như các dashboard Plotly.

## Các phân tích:
1. **Top Artists** - 10 nghệ sĩ có popularity cao nhất
2. **Genre Popularity** - Thể loại nhạc phổ biến nhất
3. **Timeline Analysis** - Xu hướng phát hành theo thời gian
4. **Market Distribution** - Phân bố theo thị trường
5. **Followers vs Popularity** - Phân tích mối quan hệ followers & popularity
6. **Top Artists by Year** - Top 5 nghệ sĩ theo từng năm
7. **Market Heatmap** - Genre popularity theo market

## 📦 Setup: Import Libraries và Khởi tạo Spark

In [1]:
# Import các thư viện cần thiết
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import pandas as pd

# Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName("Spotify Gold Layer Analysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("✅ Spark Session created successfully!")
print(f"Spark Version: {spark.version}")

ModuleNotFoundError: No module named 'pyspark'

## 📂 Load Data từ SQLite Database

In [ ]:
# Đường dẫn đến SQLite database (đã copy về local)
db_path = "spotify_gold.db"

print(f"📊 Loading data from: {db_path}")
print("⏳ Please wait...")

In [ ]:
# Vì PySpark không hỗ trợ SQLite trực tiếp, ta sẽ dùng pandas để load rồi convert sang PySpark
import sqlite3

# Kết nối SQLite
conn = sqlite3.connect(db_path)

# Load các bảng từ SQLite
tables = {
    'dim_track': 'SELECT * FROM dim_track',
    'dim_artist': 'SELECT * FROM dim_artist',
    'dim_album': 'SELECT * FROM dim_album',
    'dim_market': 'SELECT * FROM dim_market',
    'dim_genre': 'SELECT * FROM dim_genre',
    'dim_date': 'SELECT * FROM dim_date',
    'fact_track_performance': 'SELECT * FROM fact_track_performance',
    'agg_artist_performance': 'SELECT * FROM agg_artist_performance',
    'agg_market_stats': 'SELECT * FROM agg_market_stats',
    'agg_genre_popularity': 'SELECT * FROM agg_genre_popularity'
}

# Load vào pandas rồi convert sang PySpark DataFrame
spark_dfs = {}
for table_name, query in tables.items():
    pdf = pd.read_sql_query(query, conn)
    spark_dfs[table_name] = spark.createDataFrame(pdf)
    print(f"✅ Loaded {table_name}: {spark_dfs[table_name].count()} rows")

conn.close()

# Tạo temporary views cho SQL queries
for table_name, df in spark_dfs.items():
    df.createOrReplaceTempView(table_name)
    
print("\n✅ All tables loaded and registered as temp views!")

## 🎤 Query 1: Top Artists - 10 nghệ sĩ có popularity cao nhất

In [ ]:
print("="*80)
print("🎤 TOP 10 ARTISTS BY POPULARITY")
print("="*80)

# PySpark SQL Query
query1 = """
SELECT 
    artist_name,
    artist_popularity,
    artist_followers,
    total_tracks,
    ROUND(avg_popularity, 2) as avg_popularity
FROM agg_artist_performance
ORDER BY artist_popularity DESC
LIMIT 10
"""

top_artists = spark.sql(query1)
top_artists.show(truncate=False)

print("\n📊 Summary Statistics:")
top_artists.select(
    round(avg('artist_popularity'), 2).alias('avg_popularity'),
    round(avg('artist_followers'), 0).alias('avg_followers'),
    sum('total_tracks').alias('total_tracks')
).show()

## 🎵 Query 2: Genre Popularity - Thể loại nhạc phổ biến nhất

In [ ]:
print("="*80)
print("🎵 TOP GENRES BY POPULARITY")
print("="*80)

# PySpark SQL Query
query2 = """
SELECT 
    genre_name,
    ROUND(avg_popularity, 2) as avg_popularity,
    total_tracks,
    unique_artists
FROM agg_genre_popularity
ORDER BY avg_popularity DESC
LIMIT 15
"""

top_genres = spark.sql(query2)
top_genres.show(truncate=False)

print("\n📊 Genre Distribution:")
print(f"Total Genres: {spark_dfs['agg_genre_popularity'].count()}")
print(f"Total Tracks: {top_genres.agg(sum('total_tracks')).collect()[0][0]}")
print(f"Total Artists: {top_genres.agg(sum('unique_artists')).collect()[0][0]}")

## 📅 Query 3: Timeline Analysis - Xu hướng phát hành theo thời gian

In [ ]:
print("="*80)
print("📅 RELEASE TIMELINE ANALYSIS")
print("="*80)

# PySpark SQL Query - Tracks by Year and Month
query3 = """
SELECT 
    d.year,
    d.month_name,
    COUNT(DISTINCT f.track_id) as track_count,
    ROUND(AVG(f.popularity), 2) as avg_popularity
FROM fact_track_performance f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.year, d.month_name, d.month
ORDER BY d.year DESC, d.month DESC
"""

timeline = spark.sql(query3)
timeline.show(20, truncate=False)

print("\n📊 Yearly Summary:")
yearly_summary = spark.sql("""
SELECT 
    d.year,
    COUNT(DISTINCT f.track_id) as total_tracks,
    COUNT(DISTINCT f.artist_id) as total_artists,
    ROUND(AVG(f.popularity), 2) as avg_popularity
FROM fact_track_performance f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.year
ORDER BY d.year DESC
""")
yearly_summary.show(truncate=False)

## 🌍 Query 4: Market Distribution - Phân bố theo thị trường

In [ ]:
print("="*80)
print("🌍 TOP MARKETS BY PERFORMANCE")
print("="*80)

# PySpark SQL Query
query4 = """
SELECT 
    m.market_code,
    ms.total_tracks,
    ROUND(ms.avg_popularity, 2) as avg_popularity,
    ms.popular_tracks_count
FROM agg_market_stats ms
JOIN dim_market m ON ms.market_id = m.market_id
ORDER BY ms.avg_popularity DESC
LIMIT 20
"""

top_markets = spark.sql(query4)
top_markets.show(truncate=False)

print("\n📊 Market Statistics:")
market_stats = spark.sql("""
SELECT 
    COUNT(DISTINCT market_id) as total_markets,
    ROUND(AVG(avg_popularity), 2) as overall_avg_popularity,
    MAX(avg_popularity) as max_popularity,
    MIN(avg_popularity) as min_popularity
FROM agg_market_stats
""")
market_stats.show(truncate=False)

## 🔥 Query 5: Followers vs Popularity - Phân tích mối quan hệ

In [ ]:
print("="*80)
print("🔥 FOLLOWERS VS POPULARITY ANALYSIS")
print("="*80)

# PySpark SQL Query
query5 = """
SELECT 
    artist_name,
    artist_followers,
    artist_popularity,
    ROUND(avg_popularity, 2) as avg_track_popularity,
    total_tracks,
    CASE 
        WHEN artist_followers > 5000000 THEN 'Mega Star'
        WHEN artist_followers > 1000000 THEN 'Super Star'
        WHEN artist_followers > 100000 THEN 'Popular'
        ELSE 'Rising'
    END as artist_tier
FROM agg_artist_performance
ORDER BY artist_followers DESC
LIMIT 20
"""

followers_analysis = spark.sql(query5)
followers_analysis.show(truncate=False)

print("\n📊 Tier Distribution:")
tier_distribution = spark.sql("""
SELECT 
    CASE 
        WHEN artist_followers > 5000000 THEN 'Mega Star'
        WHEN artist_followers > 1000000 THEN 'Super Star'
        WHEN artist_followers > 100000 THEN 'Popular'
        ELSE 'Rising'
    END as artist_tier,
    COUNT(*) as artist_count,
    ROUND(AVG(artist_popularity), 2) as avg_popularity,
    ROUND(AVG(artist_followers), 0) as avg_followers
FROM agg_artist_performance
GROUP BY artist_tier
ORDER BY avg_followers DESC
""")
tier_distribution.show(truncate=False)

## 🏆 Query 6: Top Artists by Year - Top 5 nghệ sĩ theo từng năm

In [ ]:
print("="*80)
print("🏆 TOP 5 ARTISTS BY YEAR")
print("="*80)

# PySpark SQL Query with Window Functions
query6 = """
WITH artist_year_stats AS (
    SELECT 
        d.year,
        a.artist_name,
        COUNT(DISTINCT f.track_id) as track_count,
        ROUND(AVG(f.popularity), 2) as avg_popularity,
        MAX(a.artist_followers) as followers
    FROM fact_track_performance f
    JOIN dim_artist a ON f.artist_id = a.artist_id
    JOIN dim_date d ON f.date_id = d.date_id
    GROUP BY d.year, a.artist_name
),
ranked_artists AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY year ORDER BY avg_popularity DESC, track_count DESC) as rank
    FROM artist_year_stats
)
SELECT 
    year,
    rank,
    artist_name,
    track_count,
    avg_popularity,
    followers
FROM ranked_artists
WHERE rank <= 5
ORDER BY year DESC, rank
"""

top_by_year = spark.sql(query6)
top_by_year.show(50, truncate=False)

print("\n📊 Artists appearing multiple years:")
multi_year_artists = spark.sql("""
WITH artist_year_stats AS (
    SELECT 
        d.year,
        a.artist_name,
        ROUND(AVG(f.popularity), 2) as avg_popularity
    FROM fact_track_performance f
    JOIN dim_artist a ON f.artist_id = a.artist_id
    JOIN dim_date d ON f.date_id = d.date_id
    GROUP BY d.year, a.artist_name
)
SELECT 
    artist_name,
    COUNT(DISTINCT year) as years_active,
    ROUND(AVG(avg_popularity), 2) as overall_avg_popularity
FROM artist_year_stats
GROUP BY artist_name
HAVING COUNT(DISTINCT year) > 1
ORDER BY years_active DESC, overall_avg_popularity DESC
""")
multi_year_artists.show(10, truncate=False)

## 🗺️ Query 7: Market Heatmap - Genre popularity theo market

In [ ]:
print("="*80)
print("🗺️ MARKET-GENRE HEATMAP DATA")
print("="*80)

# PySpark SQL Query - Top genres per market
query7 = """
WITH market_genre AS (
    SELECT 
        m.market_code,
        a.primary_genres,
        COUNT(DISTINCT f.track_id) as track_count,
        ROUND(AVG(f.popularity), 2) as avg_popularity
    FROM fact_track_performance f
    JOIN dim_market m ON f.market_id = m.market_id
    JOIN dim_artist a ON f.artist_id = a.artist_id
    WHERE a.primary_genres IS NOT NULL
    GROUP BY m.market_code, a.primary_genres
),
ranked_market_genre AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY market_code ORDER BY avg_popularity DESC) as rank
    FROM market_genre
)
SELECT 
    market_code,
    primary_genres as genre,
    track_count,
    avg_popularity
FROM ranked_market_genre
WHERE rank <= 3
ORDER BY market_code, rank
"""

market_genre_heatmap = spark.sql(query7)
market_genre_heatmap.show(50, truncate=False)

print("\n📊 Most popular genres globally:")
global_genre_popularity = spark.sql("""
SELECT 
    a.primary_genres as genre,
    COUNT(DISTINCT f.market_id) as market_count,
    COUNT(DISTINCT f.track_id) as track_count,
    ROUND(AVG(f.popularity), 2) as avg_popularity
FROM fact_track_performance f
JOIN dim_artist a ON f.artist_id = a.artist_id
WHERE a.primary_genres IS NOT NULL
GROUP BY a.primary_genres
ORDER BY market_count DESC, avg_popularity DESC
LIMIT 15
""")
global_genre_popularity.show(truncate=False)

## 📈 Query 8: Advanced Analytics - Các phân tích nâng cao

In [ ]:
print("="*80)
print("📈 TRACK POPULARITY DISTRIBUTION")
print("="*80)

# Phân bố popularity của tracks
popularity_dist = spark.sql("""
SELECT 
    CASE 
        WHEN popularity >= 80 THEN 'Very High (80-100)'
        WHEN popularity >= 60 THEN 'High (60-79)'
        WHEN popularity >= 40 THEN 'Medium (40-59)'
        WHEN popularity >= 20 THEN 'Low (20-39)'
        ELSE 'Very Low (0-19)'
    END as popularity_range,
    COUNT(*) as track_count,
    ROUND(AVG(popularity), 2) as avg_popularity_in_range
FROM fact_track_performance
GROUP BY popularity_range
ORDER BY avg_popularity_in_range DESC
""")
popularity_dist.show(truncate=False)

print("\n" + "="*80)
print("🎯 COLLABORATION ANALYSIS")
print("="*80)

# Phân tích collaboration
collaboration_stats = spark.sql("""
SELECT 
    t.is_collaboration,
    COUNT(DISTINCT t.track_id) as track_count,
    ROUND(AVG(f.popularity), 2) as avg_popularity,
    COUNT(DISTINCT f.market_id) as market_reach
FROM dim_track t
JOIN fact_track_performance f ON t.track_id = f.track_id
GROUP BY t.is_collaboration
ORDER BY t.is_collaboration DESC
""")
collaboration_stats.show(truncate=False)

print("\n" + "="*80)
print("⏱️ DURATION CATEGORY ANALYSIS")
print("="*80)

# Phân tích theo duration category
duration_analysis = spark.sql("""
SELECT 
    t.duration_category,
    COUNT(DISTINCT t.track_id) as track_count,
    ROUND(AVG(t.duration_minutes), 2) as avg_duration_minutes,
    ROUND(AVG(f.popularity), 2) as avg_popularity
FROM dim_track t
JOIN fact_track_performance f ON t.track_id = f.track_id
GROUP BY t.duration_category
ORDER BY avg_popularity DESC
""")
duration_analysis.show(truncate=False)

## 🎯 Query 9: Complex Aggregations - Tổng hợp phức tạp

In [ ]:
print("="*80)
print("🎯 ARTIST PERFORMANCE METRICS")
print("="*80)

# Metrics tổng hợp của artist
artist_metrics = spark.sql("""
SELECT 
    a.artist_name,
    a.artist_popularity,
    a.artist_followers,
    COUNT(DISTINCT f.track_id) as total_tracks,
    COUNT(DISTINCT f.market_id) as market_reach,
    ROUND(AVG(f.popularity), 2) as avg_track_popularity,
    MAX(f.popularity) as max_track_popularity,
    ROUND(STDDEV(f.popularity), 2) as popularity_stddev,
    ROUND(a.artist_followers / NULLIF(COUNT(DISTINCT f.track_id), 0), 0) as followers_per_track
FROM dim_artist a
JOIN fact_track_performance f ON a.artist_id = f.artist_id
GROUP BY a.artist_name, a.artist_popularity, a.artist_followers
ORDER BY a.artist_popularity DESC
LIMIT 15
""")
artist_metrics.show(truncate=False)

print("\n" + "="*80)
print("🌟 MARKET DIVERSITY INDEX")
print("="*80)

# Diversity index của markets
market_diversity = spark.sql("""
SELECT 
    m.market_code,
    COUNT(DISTINCT f.artist_id) as unique_artists,
    COUNT(DISTINCT f.track_id) as unique_tracks,
    COUNT(DISTINCT a.primary_genres) as unique_genres,
    ROUND(AVG(f.popularity), 2) as avg_popularity,
    ROUND(COUNT(DISTINCT f.artist_id) * 1.0 / NULLIF(COUNT(DISTINCT f.track_id), 0), 3) as artist_diversity_ratio
FROM fact_track_performance f
JOIN dim_market m ON f.market_id = m.market_id
JOIN dim_artist a ON f.artist_id = a.artist_id
GROUP BY m.market_code
ORDER BY unique_artists DESC
LIMIT 20
""")
market_diversity.show(truncate=False)

## 📊 Summary Statistics - Thống kê tổng quan

In [ ]:
print("="*80)
print("📊 OVERALL SUMMARY STATISTICS")
print("="*80)

# Tổng hợp toàn bộ dữ liệu
summary = spark.sql("""
SELECT 
    'Total Records' as metric,
    COUNT(*) as value
FROM fact_track_performance
UNION ALL
SELECT 
    'Unique Tracks' as metric,
    COUNT(DISTINCT track_id) as value
FROM fact_track_performance
UNION ALL
SELECT 
    'Unique Artists' as metric,
    COUNT(DISTINCT artist_id) as value
FROM fact_track_performance
UNION ALL
SELECT 
    'Unique Markets' as metric,
    COUNT(DISTINCT market_id) as value
FROM fact_track_performance
UNION ALL
SELECT 
    'Average Popularity' as metric,
    CAST(ROUND(AVG(popularity), 2) AS BIGINT) as value
FROM fact_track_performance
UNION ALL
SELECT 
    'Max Popularity' as metric,
    CAST(MAX(popularity) AS BIGINT) as value
FROM fact_track_performance
UNION ALL
SELECT 
    'Min Popularity' as metric,
    CAST(MIN(popularity) AS BIGINT) as value
FROM fact_track_performance
UNION ALL
SELECT 
    'Total Genres' as metric,
    COUNT(*) as value
FROM agg_genre_popularity
""")
summary.show(truncate=False)

print("\n✅ Analysis Complete!")
print("🎵 All PySpark queries executed successfully!")

## 🛑 Cleanup: Dừng Spark Session

In [ ]:
# Dừng Spark session khi hoàn thành
# spark.stop()
# print("✅ Spark Session stopped successfully!")